# Phone control

## Imports

In [23]:
import subprocess
import time
import base64
import csv
import os
import random
from datetime import datetime
import pandas as pd

## Settings

In [24]:
# Keep True until the Android study phone is available.
SIMULATE = True

# Android Debug Bridge.
# Later this might need to be changed to the full path to adb.exe.
ADB_PATH = "adb"

# Package name of the Android helper app.
ANDROID_APP = "nl.sart.phonecontrol"

# Security token shared between Python and the Android helper app.
# Change this before the real experiment.
SECRET = "CHANGE_THIS_TO_SOMETHING_RANDOM"

# Experimental interruption log.
LOG_FILE = "phone_interruptions.csv"

## Create log file

In [7]:
if not os.path.exists(LOG_FILE):

    with open(LOG_FILE, "w", newline="", encoding="utf-8") as f:

        writer = csv.writer(f)

        writer.writerow([
            "participant",
            "trial",
            "interruption_type",
            "unix_time",
            "datetime"
        ])

    print(f"Created log file: {LOG_FILE}")

else:
    print(f"Log file already exists: {LOG_FILE}")

Created log file: phone_interruptions.csv


## Participant phone numbers

In [26]:
participants = {
    "P023": "+31612345678"
}

## Participant lookup

In [28]:
def get_phone(participant):

    if participant not in participants:
        raise ValueError(
            f"Participant '{participant}' does not exist "
            f"in the participant lookup table."
        )

    return participants[participant]

## Android connection

In [8]:
def adb_command(arguments):

    command = [ADB_PATH, "shell"] + arguments

    result = subprocess.run(
        command,
        capture_output=True,
        text=True
    )

    if result.returncode != 0:
        raise RuntimeError(
            f"ADB command failed:\n{result.stderr}"
        )

    return result.stdout

## Logging

In [9]:
def log_interruption(
    participant,
    trial,
    interruption_type
):

    timestamp = time.time()

    readable_time = datetime.now().isoformat(
        timespec="milliseconds"
    )

    with open(
        LOG_FILE,
        "a",
        newline="",
        encoding="utf-8"
    ) as f:

        writer = csv.writer(f)

        writer.writerow([
            participant,
            trial,
            interruption_type,
            timestamp,
            readable_time
        ])

    print(
        f"{interruption_type} triggered | "
        f"{participant} | "
        f"trial {trial} | "
        f"{readable_time}"
    )

    return timestamp

## Send SMS

In [10]:
def send_sms(
    participant,
    trial,
    phone,
    message
):

    trigger_time = log_interruption(
        participant=participant,
        trial=trial,
        interruption_type="SMS"
    )

    # -----------------------------------------------------
    # SIMULATION MODE
    # -----------------------------------------------------

    if SIMULATE:

        print(
            f"[SIMULATION] SMS → {phone}\n"
            f"Message: {message}"
        )

        return trigger_time

    # -----------------------------------------------------
    # REAL ANDROID MODE
    # -----------------------------------------------------

    # Encode the message so spaces and special characters
    # do not cause problems when sending through ADB.
    message_b64 = base64.b64encode(
        message.encode("utf-8")
    ).decode("ascii")

    adb_command([
        "am",
        "broadcast",

        "-n",
        f"{ANDROID_APP}/.CommandReceiver",

        "-a",
        "nl.sart.SEND_SMS",

        "--es",
        "token",
        SECRET,

        "--es",
        "phone",
        phone,

        "--es",
        "message_b64",
        message_b64
    ])

    print("SMS command sent to Android phone.")

    return trigger_time

## Make phone call

In [15]:
def make_call(
    participant,
    trial,
    phone
):

    trigger_time = log_interruption(
        participant=participant,
        trial=trial,
        interruption_type="CALL"
    )

    # -----------------------------------------------------
    # SIMULATION MODE
    # -----------------------------------------------------

    if SIMULATE:

        print(
            f"[SIMULATION] CALL → {phone}"
        )

        return trigger_time

    # -----------------------------------------------------
    # REAL ANDROID MODE
    # -----------------------------------------------------

    adb_command([
        "am",
        "broadcast",

        "-n",
        f"{ANDROID_APP}/.CommandReceiver",

        "-a",
        "nl.sart.MAKE_CALL",

        "--es",
        "token",
        SECRET,

        "--es",
        "phone",
        phone
    ])

    print("Call command sent to Android phone.")

    return trigger_time

## Select participant

In [19]:
participant_id = "P023"

participant_phone = participants[participant_id]

test_message = "This is a test message."

print("Participant:", participant_id)
print("Phone:", participant_phone)

Participant: P023
Phone: +31612345678


## Test SMS

In [20]:
send_sms(
    participant=participant_id,
    trial=40,
    phone=participant_phone,
    message=test_message
)

SMS triggered | P023 | trial 40 | 2026-09-22T20:48:43.724
[SIMULATION] SMS → +31612345678
Message: This is a test message.


1790102923.7241213

## Test phone call

In [21]:
make_call(
    participant=participant_id,
    trial=80,
    phone=participant_phone
)

CALL triggered | P023 | trial 80 | 2026-09-22T20:48:53.971
[SIMULATION] CALL → +31612345678


1790102933.9719598

## Look at log

In [22]:
import pandas as pd

log = pd.read_csv(LOG_FILE)

log

,participant,trial,interruption_type,unix_time,datetime
0,P023,40,SMS,1.790103e+09,2026-09-22T20:44:09.269
1,P023,80,CALL,1.790103e+09,2026-09-22T20:45:44.697
2,P023,40,SMS,1.790103e+09,2026-09-22T20:48:43.724
3,P023,80,CALL,1.790103e+09,2026-09-22T20:48:53.971
